In [ ]:
from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
import torch

from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor

model_path = "VibeVoice-1.5B"
device="mps" if torch.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
output_path = "aaa.wav"

print(f"use device[{device}]")

model = VibeVoiceForConditionalGenerationInference.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        device_map=device,
        attn_implementation='sdpa')

model.eval()
model.set_ddpm_inference_steps(num_steps=10)

full_script = "Speaker 1: 作者在本书中，围绕社会主义模式这个主题，以苏联、南斯拉夫等国的现实为依据，剖析了苏联的社会制度和苏联模式的社会主义体制的种种弊端及其矛盾，并提出了发达资本主义国家向社会主义过渡，建立真正社会主义社会的设想。"
voice_samples = "/Users/larry/coderesp/VibeVoice/demo/voices/zh-phi0_woman.WAV"

processor = VibeVoiceProcessor.from_pretrained(model_path)
inputs = processor(
    text=[full_script],  # Wrap in list for batch processing
    voice_samples=[voice_samples],  # Wrap in list for batch processing
    padding=True,
    return_tensors="pt",
    return_attention_mask=True,
)

outputs = model.generate(
    **inputs,
    max_new_tokens=None,
    cfg_scale=1.4,
    tokenizer=processor.tokenizer,
    # generation_config={'do_sample': True, 'temperature': 0.95, 'top_p': 0.95, 'top_k': 0},
    generation_config={'do_sample': False},
    verbose=True,
)

processor.save_audio(
    outputs.speech_outputs[0],  # First (and only) batch item
    output_path=output_path,
)